In [ ]:
# ============================================================
# Figure 3
# Functional interpretation of RLP3/RLP4-derived R-loop program
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, spearmanr
from sklearn.preprocessing import StandardScaler

# ============================================================
# 0. Paths
# ============================================================

PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC"

FIG1_DIR = os.path.join(PROJECT_DIR, "01_Figure1_python")
FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
FIG3_DIR = os.path.join(PROJECT_DIR, "03_Figure3_python")
os.makedirs(FIG3_DIR, exist_ok=True)

ADATA_PATH = os.path.join(FIG1_DIR, "Step10_adata_all_final_celltype.h5ad")

FIG2_META_PATH = os.path.join(
    FIG2_DIR,
    "Figure2_malignant_cell_Rloop_program_scores_metadata.csv"
)

# ============================================================
# 1. Plot settings
# ============================================================

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.unicode_minus"] = False

def save_fig(fig, name, w=5, h=4):
    fig.set_size_inches(w, h)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(FIG3_DIR, f"{name}.{ext}"),
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )
    plt.close(fig)
    print("Saved:", name)

# ============================================================
# 2. Load data
# ============================================================

adata = sc.read_h5ad(ADATA_PATH)
fig2_meta = pd.read_csv(FIG2_META_PATH, index_col=0)

common_cells = adata.obs_names.intersection(fig2_meta.index)
adata_mal = adata[common_cells].copy()

for col in fig2_meta.columns:
    adata_mal.obs[col] = fig2_meta.loc[adata_mal.obs_names, col]

adata_mal.obs["Rloop_program_group"] = pd.Categorical(
    adata_mal.obs["Rloop_program_group"],
    categories=["R-loop-program-low", "R-loop-program-high"]
)

print(adata_mal)
print(adata_mal.obs["Rloop_program_group"].value_counts())

In [ ]:
# ============================================================
# 3. Functional gene signatures Figure3A
# ============================================================

gene_sets = {
    "Cell cycle": [
        "MKI67", "TOP2A", "UBE2C", "CDK1", "CCNB1", "CCNB2",
        "CDC20", "AURKA", "AURKB", "BIRC5", "PCNA", "MCM2", "MCM4", "MCM6"
    ],
    "DNA repair": [
        "BRCA1", "BRCA2", "RAD51", "RAD51AP1", "FANCI", "FANCD2",
        "CHEK1", "CHEK2", "ATM", "ATR", "PARP1", "TOP1", "TOP2A"
    ],
    "RNA processing": [
        "SF3B1", "SRSF1", "SRSF2", "HNRNPA1", "HNRNPH1",
        "DDX5", "DDX17", "DDX39B", "RBM39", "BCLAF1", "NCL", "DHX38"
    ],
    "Hypoxia": [
        "VEGFA", "CA9", "LDHA", "ENO1", "SLC2A1", "PGK1",
        "ALDOA", "BNIP3", "NDRG1", "HIF1A"
    ],
    "Glycolysis": [
        "HK2", "PFKP", "ALDOA", "GAPDH", "PGK1", "ENO1",
        "PKM", "LDHA", "SLC2A1", "TPI1"
    ],
    "OXPHOS": [
        "NDUFA1", "NDUFA2", "NDUFB3", "NDUFS2", "COX5A",
        "COX6A1", "ATP5F1A", "ATP5F1B", "ATP5MC1", "UQCRC1"
    ],
    "EMT": [
        "VIM", "FN1", "ZEB1", "ZEB2", "SNAI1", "SNAI2",
        "TWIST1", "COL1A1", "COL1A2", "ITGA5"
    ],
    "IFN response": [
        "IFI6", "IFI27", "IFI44", "IFI44L", "ISG15", "MX1",
        "OAS1", "OAS2", "STAT1", "IRF1", "CXCL10"
    ],
    "Antigen presentation": [
        "HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2",
        "PSMB8", "PSMB9", "HLA-DRA", "HLA-DRB1"
    ]
}

for sig, genes in gene_sets.items():
    genes_use = [g for g in genes if g in adata_mal.var_names]
    print(sig, len(genes_use), genes_use)

    if len(genes_use) >= 3:
        sc.tl.score_genes(
            adata_mal,
            gene_list=genes_use,
            score_name=f"{sig}_score",
            use_raw=False
        )

score_cols = [
    f"{x}_score"
    for x in gene_sets.keys()
    if f"{x}_score" in adata_mal.obs.columns
]

# ============================================================
# Figure3A refined
# ============================================================

mean_df = adata_mal.obs.groupby("Rloop_program_group")[score_cols].mean().T

mean_z = pd.DataFrame(
    StandardScaler().fit_transform(mean_df),
    index=mean_df.index,
    columns=mean_df.columns
)

# 去掉 _score，让图更像文章图
mean_z.index = mean_z.index.str.replace("_score", "", regex=False)

# 按照生物学逻辑排序
desired_order = [
    "Cell cycle",
    "DNA repair",
    "RNA processing",
    "EMT",
    "Hypoxia",
    "Glycolysis",
    "OXPHOS",
    "IFN response",
    "Antigen presentation"
]

desired_order = [x for x in desired_order if x in mean_z.index]
mean_z = mean_z.loc[desired_order]

# 保证 low 在左边，high 在右边
desired_cols = [
    "R-loop-program-low",
    "R-loop-program-high"
]
desired_cols = [x for x in desired_cols if x in mean_z.columns]
mean_z = mean_z[desired_cols]

fig, ax = plt.subplots(figsize=(4.8, 6.5))

sns.heatmap(
    mean_z,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Scaled mean score"}
)

ax.set_title(
    "Functional states of R-loop-program groups",
    fontsize=14,
    fontweight="bold",
    pad=12
)

ax.set_xlabel("")
ax.set_ylabel("Functional signatures", fontsize=12)

# 横坐标倾斜排列
plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

plt.setp(ax.get_yticklabels(), rotation=0)

ax.tick_params(axis="both", labelsize=11)

save_fig(
    fig,
    "Figure3A_Functional_signature_heatmap_refined",
    w=4.8,
    h=6.5
)

In [ ]:
# ============================================================
# Figure3B
# Key functional signatures between high and low
# with matched colors and significance labels
# ============================================================

from scipy.stats import mannwhitneyu

key_scores = [
    "Cell cycle_score",
    "DNA repair_score",
    "RNA processing_score",
    "Hypoxia_score",
    "Glycolysis_score",
    "OXPHOS_score",
    "IFN response_score",
    "Antigen presentation_score"
]

key_scores = [x for x in key_scores if x in adata_mal.obs.columns]

plot_df = adata_mal.obs[["Rloop_program_group"] + key_scores].copy()

plot_long = plot_df.melt(
    id_vars="Rloop_program_group",
    value_vars=key_scores,
    var_name="Signature",
    value_name="Score"
)

plot_long["Signature"] = plot_long["Signature"].str.replace("_score", "", regex=False)

# 按照生物学逻辑排序
signature_order = [
    "Cell cycle",
    "DNA repair",
    "RNA processing",
    "Hypoxia",
    "Glycolysis",
    "OXPHOS",
    "IFN response",
    "Antigen presentation"
]
signature_order = [x for x in signature_order if x in plot_long["Signature"].unique()]

plot_long["Signature"] = pd.Categorical(
    plot_long["Signature"],
    categories=signature_order,
    ordered=True
)

# 与 Figure2F 保持一致：high 蓝色，low 橙色
group_order = ["R-loop-program-high", "R-loop-program-low"]
palette = {
    "R-loop-program-high": "#1f77b4",
    "R-loop-program-low": "#ff7f0e"
}

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# 统计检验
stat_results = {}
for sig in signature_order:
    df_sig = plot_long[plot_long["Signature"] == sig]

    high_vals = df_sig.loc[
        df_sig["Rloop_program_group"] == "R-loop-program-high",
        "Score"
    ].dropna()

    low_vals = df_sig.loc[
        df_sig["Rloop_program_group"] == "R-loop-program-low",
        "Score"
    ].dropna()

    if len(high_vals) > 0 and len(low_vals) > 0:
        _, p = mannwhitneyu(high_vals, low_vals, alternative="two-sided")
    else:
        p = np.nan

    stat_results[sig] = {
        "p": p,
        "star": p_to_star(p) if not np.isnan(p) else "NA"
    }

# 画图
fig, ax = plt.subplots(figsize=(10.5, 4.5))

sns.violinplot(
    data=plot_long,
    x="Signature",
    y="Score",
    hue="Rloop_program_group",
    order=signature_order,
    hue_order=group_order,
    palette=palette,
    split=True,
    inner=None,
    cut=0,
    linewidth=1,
    ax=ax
)

# 添加中位数点，增强可读性
sns.pointplot(
    data=plot_long,
    x="Signature",
    y="Score",
    hue="Rloop_program_group",
    order=signature_order,
    hue_order=group_order,
    palette=palette,
    dodge=0.25,
    estimator=np.median,
    errorbar=None,
    markers=".",
    markersize=7,
    linestyles="",
    ax=ax
)

# 去除 pointplot 产生的重复 legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# 添加显著性标记
y_min, y_max = ax.get_ylim()
y_range = y_max - y_min

for i, sig in enumerate(signature_order):
    df_sig = plot_long[plot_long["Signature"] == sig]
    y = df_sig["Score"].max()
    y_bar = y + 0.06 * y_range
    y_text = y + 0.09 * y_range

    ax.plot(
        [i - 0.22, i - 0.22, i + 0.22, i + 0.22],
        [y_bar, y_text, y_text, y_bar],
        lw=1,
        color="black"
    )

    ax.text(
        i,
        y_text + 0.01 * y_range,
        stat_results[sig]["star"],
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# 重新调整 y 轴，避免星号被截断
ax.set_ylim(y_min, y_max + 0.18 * y_range)

ax.set_xlabel("")
ax.set_ylabel("Signature score", fontsize=12)
ax.set_title(
    "Functional signature scores in R-loop-program groups",
    fontsize=14,
    fontweight="bold",
    pad=12
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor"
)

ax.tick_params(axis="both", labelsize=11)

save_fig(
    fig,
    "Figure3B_Key_signature_violin_refined",
    w=10.5,
    h=4.5
)

In [ ]:
!pip install adjustText

In [ ]:
# ============================================================
# 4. Differential expression: high vs low
# ============================================================

sc.tl.rank_genes_groups(
    adata_mal,
    groupby="Rloop_program_group",
    groups=["R-loop-program-high"],
    reference="R-loop-program-low",
    method="wilcoxon",
    use_raw=False
)

de = sc.get.rank_genes_groups_df(
    adata_mal,
    group="R-loop-program-high"
)

# 避免无穷大
de["pvals_adj"] = de["pvals_adj"].replace(0, 1e-300)
de["minus_log10_padj"] = -np.log10(de["pvals_adj"])

# 阈值
logfc_cutoff = 0.25
padj_cutoff = 0.05

de["direction"] = "NS"
de.loc[
    (de["logfoldchanges"] > logfc_cutoff) & (de["pvals_adj"] < padj_cutoff),
    "direction"
] = "High-up"

de.loc[
    (de["logfoldchanges"] < -logfc_cutoff) & (de["pvals_adj"] < padj_cutoff),
    "direction"
] = "Low-up"

de.to_csv(
    os.path.join(FIG3_DIR, "Figure3C_DE_Rloop_program_high_vs_low.csv"),
    index=False
)

# ============================================================
# Figure3C volcano plot refined
# ============================================================

from adjustText import adjust_text

palette = {
    "High-up": "#D73027",   # red
    "Low-up": "#4575B4",    # blue
    "NS": "#BDBDBD"         # grey
}

plot_order = ["NS", "Low-up", "High-up"]

fig, ax = plt.subplots(figsize=(6.2, 5.2))

# 分层画点：先画灰色，再画显著点，避免显著点被遮挡
for group in plot_order:
    df_g = de[de["direction"] == group]

    ax.scatter(
        df_g["logfoldchanges"],
        df_g["minus_log10_padj"],
        s=14 if group != "NS" else 8,
        c=palette[group],
        alpha=0.85 if group != "NS" else 0.45,
        linewidths=0,
        label=group
    )

# 阈值线
ax.axvline(logfc_cutoff, linestyle="--", linewidth=1, color="black", alpha=0.55)
ax.axvline(-logfc_cutoff, linestyle="--", linewidth=1, color="black", alpha=0.55)
ax.axhline(-np.log10(padj_cutoff), linestyle="--", linewidth=1, color="black", alpha=0.55)

# 选择需要标注的基因：左右两边各选一部分
top_high = (
    de[de["direction"] == "High-up"]
    .sort_values(["pvals_adj", "logfoldchanges"], ascending=[True, False])
    .head(6)
)

top_low = (
    de[de["direction"] == "Low-up"]
    .sort_values(["pvals_adj", "logfoldchanges"], ascending=[True, True])
    .head(6)
)

top_label = pd.concat([top_high, top_low], axis=0)

texts = []
for _, row in top_label.iterrows():
    texts.append(
        ax.text(
            row["logfoldchanges"],
            row["minus_log10_padj"],
            row["names"],
            fontsize=8,
            color="black"
        )
    )

# 自动避让标签 + 箭头
adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(
        arrowstyle="-",
        color="black",
        lw=0.6,
        alpha=0.8
    ),
    expand_points=(1.4, 1.6),
    expand_text=(1.2, 1.4),
    force_points=0.3,
    force_text=0.5,
    lim=500
)

# 阈值说明
threshold_text = (
    f"|log2FC| > {logfc_cutoff}\n"
    f"adjusted P < {padj_cutoff}"
)

ax.text(
    0.98,
    0.96,
    threshold_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="grey",
        alpha=0.85
    )
)

ax.set_xlabel("log2 fold change", fontsize=12)
ax.set_ylabel("-log10 adjusted P-value", fontsize=12)
ax.set_title(
    "R-loop-program-high vs low",
    fontsize=14,
    fontweight="bold",
    pad=10
)

ax.legend(
    frameon=False,
    fontsize=9,
    loc="upper left",
    title=None
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=10)

save_fig(
    fig,
    "Figure3C_Volcano_high_vs_low_refined",
    w=6.2,
    h=5.2
)

In [ ]:
!pip install gseapy

In [ ]:
# ============================================================
# Figure3D
# Functional enrichment of DE genes - refined version
# ============================================================

import re
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ----------------------------
# 1. Select High-up genes
# ----------------------------

up_genes = de.loc[
    de["direction"] == "High-up",
    "names"
].dropna().astype(str).tolist()

up_genes = list(dict.fromkeys(up_genes))[:300]

print("Number of High-up genes:", len(up_genes))
print(up_genes[:20])

# ----------------------------
# 2. Enrichment analysis
# ----------------------------

enr = gp.enrichr(
    gene_list=up_genes,
    gene_sets=[
        "GO_Biological_Process_2021",
        "KEGG_2021_Human",
        "MSigDB_Hallmark_2020"
    ],
    organism="human",
    outdir=None,
    cutoff=0.5
)

enr_df = enr.results.copy()

# ----------------------------
# 3. Calculate enrichment metrics
# ----------------------------

enr_df["minus_log10_padj"] = -np.log10(enr_df["Adjusted P-value"] + 1e-300)

# 解析 Overlap，例如 18/300
def parse_overlap(x):
    try:
        a, b = str(x).split("/")
        return int(a), int(b), int(a) / int(b)
    except:
        return np.nan, np.nan, np.nan

overlap_parsed = enr_df["Overlap"].apply(parse_overlap)
enr_df["overlap_count"] = [x[0] for x in overlap_parsed]
enr_df["term_size"] = [x[1] for x in overlap_parsed]
enr_df["gene_ratio"] = [x[2] for x in overlap_parsed]

# ----------------------------
# 4. Clean pathway names
# ----------------------------

def clean_term(term):
    term = re.sub(r"\s*\(GO:\d+\)", "", term)
    term = term.replace("HALLMARK_", "")
    term = term.replace("_", " ")
    term = term.replace("KEGG ", "")
    term = term.title()

    replacements = {
        "Myc Targets V1": "MYC targets",
        "G2-M Checkpoint": "G2M checkpoint",
        "G2M Checkpoint": "G2M checkpoint",
        "E2F Targets": "E2F targets",
        "Mtorc1 Signaling": "mTORC1 signaling",
        "Estrogen Response Late": "Estrogen response",
        "Mrna": "mRNA",
        "Rna": "RNA",
        "Via": "via"
    }

    for k, v in replacements.items():
        term = term.replace(k, v)

    return term

enr_df["Term_clean"] = enr_df["Term"].apply(clean_term)

# ----------------------------
# 5. Select representative pathways
# ----------------------------

keywords = [
    "MYC",
    "G2M",
    "E2F",
    "mRNA",
    "RNA",
    "Spliceosome",
    "Splicing",
    "Processing",
    "Mitotic",
    "Stress",
    "Translation",
    "mTORC1"
]

enr_focus = enr_df[
    enr_df["Term_clean"].str.contains("|".join(keywords), case=False, na=False)
].copy()

# 去掉特别重复的长条目，保留更有代表性的 term
drop_patterns = [
    "negative regulation of RNA splicing",
    "negative regulation of mRNA splicing",
    "negative regulation of mRNA processing",
    "RNA splicing, via transesterification"
]

for pat in drop_patterns:
    enr_focus = enr_focus[
        ~enr_focus["Term"].str.contains(pat, case=False, na=False)
    ]

# 如果过滤后太少，就用显著性前15个补充
if enr_focus.shape[0] < 8:
    enr_focus = enr_df.copy()

enr_plot = (
    enr_focus
    .sort_values("Adjusted P-value", ascending=True)
    .head(12)
    .copy()
)

# 为了图上从上到下按照显著性排序
enr_plot = enr_plot.sort_values("minus_log10_padj", ascending=True)

enr_plot.to_csv(
    os.path.join(FIG3_DIR, "Figure3D_Enrichment_high_up_genes_refined.csv"),
    index=False
)

# ----------------------------
# 6. Plot refined dotplot
# ----------------------------

fig, ax = plt.subplots(figsize=(6.8, 4.8))

scatter = sns.scatterplot(
    data=enr_plot,
    x="minus_log10_padj",
    y="Term_clean",
    size="overlap_count",
    hue="gene_ratio",
    sizes=(45, 220),
    palette="viridis",
    edgecolor="black",
    linewidth=0.3,
    ax=ax
)

ax.set_xlabel("-log10 adjusted P-value", fontsize=12)
ax.set_ylabel("")
ax.set_title(
    "Pathway enrichment of R-loop-program-high genes",
    fontsize=14,
    fontweight="bold",
    pad=10
)

ax.tick_params(axis="both", labelsize=10)

# 图例更清楚
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles,
    labels=labels,
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    title=None
)

# 注释阈值/输入基因
ax.text(
    0.98,
    0.03,
    f"Top {len(up_genes)} high-up genes",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(
    fig,
    "Figure3D_Enrichment_dotplot_high_up_refined",
    w=6.8,
    h=4.8
)

In [ ]:
# ============================================================
# Figure3E
# Proportion of high/low malignant cells per sample
# with malignant cell number panel
# ============================================================

import matplotlib.gridspec as gridspec

possible_sample_cols = ["sample", "Sample", "sample_id", "orig.ident", "dataset"]
sample_col = None

for col in possible_sample_cols:
    if col in adata_mal.obs.columns:
        sample_col = col
        break

if sample_col is None:
    raise ValueError("No sample column found. Please check adata_mal.obs columns.")

print("Using sample column:", sample_col)

# ----------------------------
# 1. Count high/low cells per sample
# ----------------------------

prop_df = (
    adata_mal.obs
    .groupby([sample_col, "Rloop_program_group"])
    .size()
    .reset_index(name="n")
)

prop_df["total"] = prop_df.groupby(sample_col)["n"].transform("sum")
prop_df["proportion"] = prop_df["n"] / prop_df["total"] * 100

# ----------------------------
# 2. Wide table
# ----------------------------

prop_wide = prop_df.pivot(
    index=sample_col,
    columns="Rloop_program_group",
    values="proportion"
).fillna(0)

count_wide = prop_df.pivot(
    index=sample_col,
    columns="Rloop_program_group",
    values="n"
).fillna(0)

count_wide["Total_malignant_cells"] = count_wide.sum(axis=1)
count_wide["log2_malignant_cell_no"] = np.log2(count_wide["Total_malignant_cells"] + 1)

# ----------------------------
# 3. Sort samples by high proportion
# ----------------------------

if "R-loop-program-high" in prop_wide.columns:
    sample_order = (
        prop_wide
        .sort_values("R-loop-program-high", ascending=False)
        .index
        .tolist()
    )
else:
    sample_order = prop_wide.index.tolist()

prop_wide = prop_wide.loc[sample_order]
count_wide = count_wide.loc[sample_order]

# ----------------------------
# 4. Save tables
# ----------------------------

prop_wide.to_csv(
    os.path.join(FIG3_DIR, "Figure3E_Rloop_program_group_proportion_per_sample.csv")
)

count_wide.to_csv(
    os.path.join(FIG3_DIR, "Figure3E_Rloop_program_group_cell_counts_per_sample.csv")
)

# ----------------------------
# 5. Plot
# ----------------------------

palette = {
    "R-loop-program-high": "#1f77b4",  # blue, same as Figure2F
    "R-loop-program-low": "#ff7f0e"    # orange, same as Figure2F
}

fig = plt.figure(figsize=(12, 5.5))
gs = gridspec.GridSpec(
    2,
    1,
    height_ratios=[1, 3],
    hspace=0.05
)

ax_top = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1], sharex=ax_top)

x = np.arange(len(sample_order))

# ----------------------------
# Top panel: malignant cell number
# ----------------------------

ax_top.plot(
    x,
    count_wide["log2_malignant_cell_no"].values,
    color="black",
    marker="o",
    markersize=3,
    linewidth=1
)

ax_top.set_ylabel("log2\nmalignant\ncell no.", fontsize=10)
ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)
ax_top.tick_params(axis="x", labelbottom=False)

# ----------------------------
# Bottom panel: stacked proportion
# ----------------------------

bottom = np.zeros(len(sample_order))

# 为了和 Figure2F 颜色一致，这里 high 蓝色，low 橙色
for group in ["R-loop-program-high", "R-loop-program-low"]:
    if group in prop_wide.columns:
        ax_bar.bar(
            x,
            prop_wide[group].values,
            bottom=bottom,
            label=group,
            color=palette[group],
            width=0.85
        )
        bottom += prop_wide[group].values

ax_bar.set_ylabel("Malignant cell proportion (%)", fontsize=11)
ax_bar.set_xlabel("")
ax_bar.set_ylim(0, 100)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(sample_order, rotation=90, fontsize=8)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)

ax_bar.legend(
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10
)

fig.suptitle(
    "Composition of R-loop-program groups across samples",
    fontsize=14,
    fontweight="bold",
    y=0.98
)

save_fig(
    fig,
    "Figure3E_Rloop_program_composition_with_cell_number",
    w=12,
    h=5.5
)

In [ ]:
# ============================================================
# Figure3E
# Proportion of high/low malignant cells per sample
# ============================================================

possible_sample_cols = ["sample", "Sample", "sample_id", "orig.ident", "dataset"]
sample_col = None

for col in possible_sample_cols:
    if col in adata_mal.obs.columns:
        sample_col = col
        break

print("Using sample column:", sample_col)

prop_df = (
    adata_mal.obs
    .groupby([sample_col, "Rloop_program_group"])
    .size()
    .reset_index(name="n")
)

prop_df["total"] = prop_df.groupby(sample_col)["n"].transform("sum")
prop_df["proportion"] = prop_df["n"] / prop_df["total"] * 100

prop_wide = prop_df.pivot(
    index=sample_col,
    columns="Rloop_program_group",
    values="proportion"
).fillna(0)

if "R-loop-program-high" in prop_wide.columns:
    prop_wide = prop_wide.sort_values("R-loop-program-high", ascending=False)

prop_wide.to_csv(
    os.path.join(FIG3_DIR, "Figure3E_Rloop_program_group_proportion_per_sample.csv")
)

fig, ax = plt.subplots(figsize=(10, 4))

bottom = np.zeros(prop_wide.shape[0])

for group in ["R-loop-program-low", "R-loop-program-high"]:
    if group in prop_wide.columns:
        ax.bar(
            prop_wide.index.astype(str),
            prop_wide[group],
            bottom=bottom,
            label=group
        )
        bottom += prop_wide[group].values

ax.set_ylabel("Malignant cell proportion (%)")
ax.set_xlabel("")
ax.set_title("Proportion of R-loop-program groups in each sample")
ax.tick_params(axis="x", rotation=90)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

save_fig(fig, "Figure3E_Proportion_high_low_per_sample", w=10, h=4)

In [ ]:
# ============================================================
# Figure3F
# Correlation between RLP3/RLP4 combined score and functional states
# ============================================================

corr_rows = []

target_score = "RLP3_RLP4_combined_score"

for sig in score_cols:
    df_tmp = adata_mal.obs[[target_score, sig]].dropna()
    r, p = spearmanr(df_tmp[target_score], df_tmp[sig])
    corr_rows.append({
        "Signature": sig.replace("_score", ""),
        "Spearman_r": r,
        "P_value": p
    })

corr_df = pd.DataFrame(corr_rows)
corr_df["minus_log10_p"] = -np.log10(corr_df["P_value"] + 1e-300)

corr_df.to_csv(
    os.path.join(FIG3_DIR, "Figure3F_RLP_combined_score_function_correlation.csv"),
    index=False
)

corr_mat = corr_df.set_index("Signature")[["Spearman_r"]]

fig, ax = plt.subplots(figsize=(3, 5))
sns.heatmap(
    corr_mat,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"label": "Spearman r"}
)

ax.set_title("Correlation with RLP3/RLP4 combined score")
ax.set_xlabel("")

save_fig(fig, "Figure3F_Correlation_RLP_combined_function", w=3, h=5)

In [ ]:
# ============================================================
# Figure3G
# Continuous association between RLP3/RLP4 combined score
# and key functional signatures
# ============================================================

from scipy.stats import spearmanr

target_score = "RLP3_RLP4_combined_score"

selected_scores = [
    "EMT_score",
    "IFN response_score",
    "Antigen presentation_score"
]

selected_scores = [x for x in selected_scores if x in adata_mal.obs.columns]

plot_names = {
    "EMT_score": "EMT",
    "IFN response_score": "IFN response",
    "Antigen presentation_score": "Antigen presentation"
}

fig, axes = plt.subplots(
    1,
    len(selected_scores),
    figsize=(4.2 * len(selected_scores), 3.8),
    sharex=False,
    sharey=False
)

if len(selected_scores) == 1:
    axes = [axes]

for ax, score_col in zip(axes, selected_scores):
    df_plot = adata_mal.obs[[target_score, score_col]].dropna().copy()

    r, p = spearmanr(df_plot[target_score], df_plot[score_col])

    sns.regplot(
        data=df_plot,
        x=target_score,
        y=score_col,
        scatter_kws={
            "s": 8,
            "alpha": 0.35,
            "linewidths": 0
        },
        line_kws={
            "linewidth": 1.5,
            "color": "black"
        },
        lowess=True,
        ax=ax
    )

    ax.set_xlabel("RLP3/RLP4 combined score", fontsize=10)
    ax.set_ylabel(plot_names.get(score_col, score_col.replace("_score", "")), fontsize=10)

    ax.text(
        0.05,
        0.95,
        f"Spearman r = {r:.2f}\nP = {p:.1e}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="grey",
            alpha=0.85
        )
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Continuous association between R-loop program activity and functional states",
    fontsize=14,
    fontweight="bold",
    y=1.05
)

plt.tight_layout()

save_fig(
    fig,
    "Figure3G_RLP_combined_score_function_scatter",
    w=4.2 * len(selected_scores),
    h=3.8
)

In [ ]:
adata_mal.write(
    os.path.join(FIG3_DIR, "Figure3_malignant_cells_with_function_scores.h5ad")
)

adata_mal.obs.to_csv(
    os.path.join(FIG3_DIR, "Figure3_malignant_cells_metadata_with_function_scores.csv")
)

print("Figure3 finished.")
print("Output:", FIG3_DIR)